# Day 031 — Exercise 1: retry

**What you'll build:** `retry(fn, max_attempts=3, base_delay=1.0, backoff=2.0)` — calls `fn()` up to `max_attempts` times; on each failure sleeps `base_delay * (backoff ** attempt)` seconds before the next try; raises the LAST exception if all attempts fail.

**Why it matters:** Network calls, API requests, and file I/O fail intermittently. A retry loop turns transient failures into automatic recovery. Exponential backoff prevents hammering a struggling service with immediate retries.

In [ ]:
import time

## Your Implementation

In [ ]:
def retry(
    fn,
    max_attempts: int = 3,
    base_delay: float = 1.0,
    backoff: float = 2.0,
):
    """
    Call fn() up to max_attempts times with exponential backoff.

    Args:
        fn:           Zero-arg callable to retry.
        max_attempts: Maximum number of attempts (default 3).
        base_delay:   Seconds to wait after first failure (default 1.0).
        backoff:      Multiplier applied each subsequent retry (default 2.0).

    Returns:
        Return value of fn() if any attempt succeeds.

    Raises:
        The exception from the LAST failed attempt.
    """
    # TODO: last_error = None
    # TODO: for attempt in range(max_attempts):
    #     try: return fn()
    #     except Exception as e:
    #         last_error = e
    #         if attempt < max_attempts - 1: time.sleep(base_delay * (backoff ** attempt))
    # TODO: raise last_error
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined
    try:
        assert 'retry' in globals()
        passed += 1; print('\u2705 Check 1: retry defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}')
        return

    # Check 2: succeeds on first try → correct return value
    try:
        result = retry(lambda: 42, max_attempts=3, base_delay=0.0)
        assert result == 42, f'expected 42, got {result!r}'
        passed += 1; print('\u2705 Check 2: immediate success returns correct value')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: fn fails twice then succeeds → returns correct result
    try:
        _n = [0]
        def _flaky():
            _n[0] += 1
            if _n[0] < 3:
                raise ValueError(f'attempt {_n[0]} failed')
            return 'success'
        result = retry(_flaky, max_attempts=3, base_delay=0.0)
        assert result == 'success', f'expected success, got {result!r}'
        assert _n[0] == 3, f'fn should be called 3 times, got {_n[0]}'
        passed += 1; print('\u2705 Check 3: retries until success on 3rd attempt')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: always fails → raises LAST exception (not first)
    try:
        _m = [0]
        def _always_fail():
            _m[0] += 1
            raise RuntimeError(f'error #{_m[0]}')
        raised = False
        try:
            retry(_always_fail, max_attempts=3, base_delay=0.0)
        except RuntimeError as e:
            raised = True
            assert 'error #3' in str(e), \
                f'should raise LAST error (#3), got: {e}'
        assert raised, 'retry should have raised RuntimeError'
        passed += 1; print('\u2705 Check 4: raises the LAST exception after all retries')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: fn called exactly max_attempts times when always failing
    try:
        _k = [0]
        def _count_calls():
            _k[0] += 1
            raise ValueError('always fails')
        try:
            retry(_count_calls, max_attempts=4, base_delay=0.0)
        except ValueError:
            pass
        assert _k[0] == 4, f'expected 4 calls (max_attempts=4), got {_k[0]}'
        passed += 1; print(f'\u2705 Check 5: fn called exactly max_attempts times')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
import time


def retry(fn, max_attempts: int = 3, base_delay: float = 1.0, backoff: float = 2.0):
    last_error = None
    for attempt in range(max_attempts):
        try:
            return fn()
        except Exception as e:
            last_error = e
            if attempt < max_attempts - 1:
                time.sleep(base_delay * (backoff ** attempt))
    raise last_error
```

</details>